# Week 8 — Walk-Forward Forecasting and Backtest Integrity

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Implement time-based train/test splits and walk-forward evaluation.
- Build a forecast model and compare it against naive baselines.
- Apply trading costs and compare gross vs net performance.
- Deliberately exhibit a leaked model and explain why its results are invalid.

## Estimated study time

About 10–12 hours.

## Prerequisites

- Time series from Week 7
- Regression from Week 5

## External resources

- [Forecasting: Principles and Practice, the Pythonic Way](https://otexts.com/fpppy/)
- [Penn State STAT 510 Applied Time Series Analysis](https://online.stat.psu.edu/stat510/)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

This week assembles the previous seven weeks into one **correct** research workflow. The core principles:

1. **Time-based splits**: the training set always comes before the test set.
2. **Walk-forward evaluation**: expanding (anchored start, growing window) or rolling (fixed length, sliding forward).
3. **No leakage**: features at time $t$ may only use information up to $t$; positions driven by a signal must be **correctly lagged** to line up with future returns.
4. **Trading costs**: compare gross vs net; only net is the honest result.
5. **Baselines**: a model that cannot beat a naive baseline has no value.

> This notebook is **methodology training**, not an investable strategy. No result here represents real-world profitability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_prices
from quant_math_roadmap.finance.returns import simple_returns
from quant_math_roadmap.time_series.splits import (
    expanding_window_splits, train_test_split_time,
)
from quant_math_roadmap.time_series.forecasting import (
    fit_linear_lag_model, forecast_error_metrics,
    historical_mean_forecast, zero_forecast,
)
from quant_math_roadmap.backtesting.engine import (
    buy_and_hold_benchmark, information_coefficient, run_backtest,
)
from quant_math_roadmap.backtesting.leakage_checks import (
    assert_no_lookahead, leaked_strategy_returns, signal_to_positions,
)
from quant_math_roadmap.backtesting.costs import cost_summary

config = SyntheticConfig(n_assets=1, n_periods=900, seed=33)
prices = generate_correlated_prices(config).iloc[:, 0]
returns = simple_returns(prices)
print('Return series length:', len(returns))

### A time-based train/test split

In [ ]:
split = train_test_split_time(len(returns), test_size=0.3)
train = returns.iloc[split.train_index]
test = returns.iloc[split.test_index]
print(f'Train: {len(train)} periods | Test: {len(test)} periods')
print('Last train day <', 'first test day:',
      train.index[-1] < test.index[0])

### Purged splits: a firebreak between train and test

When features or labels **span multiple periods** (rolling features, multi-day forward returns), the tail of the training set and the start of the test set actually **share the same information** — even with a strictly chronological split, the boundary still leaks. The fix is 'purging': leave a **gap** between train and test at least as long as the information overlap. Both `expanding_window_splits` and `rolling_window_splits` support a `gap` argument.

In [ ]:
with_gap = list(expanding_window_splits(
    len(returns), initial_train_size=600, test_size=50, gap=10))
no_gap = list(expanding_window_splits(
    len(returns), initial_train_size=600, test_size=50, gap=0))
s_gap, s_plain = with_gap[0], no_gap[0]
print(f'No gap: train ends at {s_plain.train_index.max()}, '
      f'test starts at {s_plain.test_index.min()}')
print(f'gap=10: train ends at {s_gap.train_index.max()}, '
      f'test starts at {s_gap.test_index.min()}  <- the 10 periods between are used by neither side')
print('If a feature uses a 10-period rolling window, gap >= 10 keeps the boundary clean.')

### Forecast model vs naive baselines

In [ ]:
# Honest one-step-ahead forecasts via a walk-forward expanding window
predictions, actuals, baseline_mean, baseline_zero = [], [], [], []
for sp in expanding_window_splits(len(returns), initial_train_size=400,
                                  test_size=1):
    tr = returns.iloc[sp.train_index]
    te = returns.iloc[sp.test_index]
    # Simple linear lag forecast: most recent return times the train-set lag-1 autocorrelation
    lag1 = tr.autocorr(lag=1)
    pred = lag1 * tr.iloc[-1]
    predictions.append(pred)
    actuals.append(te.iloc[0])
    baseline_mean.append(historical_mean_forecast(tr))
    baseline_zero.append(zero_forecast(tr))

actual_s = pd.Series(actuals)
print('Lag forecast    :', forecast_error_metrics(actual_s, pd.Series(predictions)))
print('Historical mean :', forecast_error_metrics(actual_s, pd.Series(baseline_mean)))
print('Naive zero      :', forecast_error_metrics(actual_s, pd.Series(baseline_zero)))

On synthetic (near white-noise) returns, the forecast model **usually fails to beat** the naive baselines. That is a healthy outcome: it honestly reflects how hard returns are to predict.

### Quantifying forecast power with a multi-lag linear model + the information coefficient (IC)

The lag-1 forecast above was hand-rolled. `fit_linear_lag_model` fits $y_t = c + \sum_k b_k\, x_{t-k}$ in one go; `information_coefficient` computes the correlation between the **signal** and **future returns** — a single number that tells you whether the forecast has any directional skill.

In [ ]:
# Fit a 3-lag linear model on the training set, then forecast the test set period by period
train = returns.iloc[:600]
test = returns.iloc[600:]
lag_model = fit_linear_lag_model(train, n_lags=3)
print('Coefficients [intercept, lag1, lag2, lag3] =', np.round(lag_model.coefficients, 6))

# Build test-set forecasts from the sliding window of the 3 most recent historical returns
rolling_lags = pd.concat([returns.shift(k) for k in (1, 2, 3)], axis=1)
rolling_lags = rolling_lags.loc[test.index]
preds = pd.Series(
    [lag_model.predict(row.to_numpy()) for _, row in rolling_lags.iterrows()],
    index=test.index,
)
ic = information_coefficient(preds, test)
print(f'Test-set information coefficient (IC) = {ic:.4f}')
print('|IC| near 0 is expected: synthetic daily returns are essentially white noise.')

### Turning signals into positions (lagged correctly to avoid leakage)

In [ ]:
# Signal: the sign of yesterday's return. Positions must be lagged 1 period before trading.
raw_signal = np.sign(returns)
positions = signal_to_positions(raw_signal, lag=1)
print('First 5 signals  :', raw_signal.head().to_list())
print('First 5 positions:', positions.head().to_list())
print('Positions are the lagged signal — day one is 0 (no usable information yet).')

### Gross vs net: the impact of trading costs

In [ ]:
result = run_backtest(raw_signal, returns, signal_lag=1,
                      cost_per_unit_turnover=0.0005)
summary = result.summary()
for k, v in summary.items():
    print(f'{k}: {v:.6f}')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(result.gross_equity.index, result.gross_equity.values,
        label='gross (before costs)')
ax.plot(result.net_equity.index, result.net_equity.values,
        label='net (after costs)')
bnh = buy_and_hold_benchmark(returns)
ax.plot(bnh.index, bnh.values, label='buy-and-hold benchmark', linestyle='--')
ax.set_title('Backtest equity curves: gross vs net vs benchmark')
ax.set_xlabel('Date')
ax.set_ylabel('Equity (start = 1)')
ax.legend()
plt.show()

Trading costs open a clear gap between gross and net. **A backtest that only shows gross is not honest.**

### The multi-asset version: backtesting a whole portfolio from a weight schedule

Every discipline from the single-asset engine — lagging, costs, gross vs net — applies just the same with multiple assets. `run_portfolio_backtest()` takes a table of **target weights** (each row computed from information known that day), lags it one period automatically, and charges costs on the change in weights.

In [ ]:
from quant_math_roadmap.backtesting import run_portfolio_backtest
panel_cfg = SyntheticConfig(n_assets=4, n_periods=900, seed=44,
                            average_correlation=0.3)
panel_prices = generate_correlated_prices(panel_cfg)
panel_returns = simple_returns(panel_prices)

# Simple demo: fixed equal weights vs 30-day momentum-tilted weights
eq_weights = pd.DataFrame(0.25, index=panel_returns.index,
                          columns=panel_returns.columns)
momentum = panel_returns.rolling(30).mean()
tilt = momentum.rank(axis=1)  # momentum rank as the basis for the weights
tilt = tilt.div(tilt.sum(axis=1), axis=0).fillna(0.0)

res_eq = run_portfolio_backtest(eq_weights, panel_returns,
                                cost_per_unit_turnover=0.0005)
res_tilt = run_portfolio_backtest(tilt, panel_returns,
                                  cost_per_unit_turnover=0.0005)
print('Equal weight  :', {k: round(v, 4) for k, v in res_eq.summary().items()})
print('Momentum tilt :', {k: round(v, 4) for k, v in res_tilt.summary().items()})
print('Note the clearly higher avg_turnover and cost_drag of the momentum version.')

### Parameter sweeps: watching curve-fitting happen

The final lesson: take a strategy with a single knob (the trailing-momentum lookback), run one backtest per candidate value, and plot the **in-sample** and **out-of-sample** Sharpe side by side. The parameter with the best IS Sharpe is usually unremarkable OOS — that gap is the price of curve-fitting.

In [ ]:
from quant_math_roadmap.backtesting import lookback_parameter_sweep

lookbacks = [3, 5, 8, 13, 21, 34, 55, 89]
sweep = lookback_parameter_sweep(returns, lookbacks,
                                 in_sample_fraction=0.6,
                                 cost_per_unit_turnover=0.0005)
print(sweep[['is_sharpe', 'oos_sharpe']].round(3))
best_is = sweep['is_sharpe'].idxmax()
print(f'Best IS lookback = {best_is}, its OOS Sharpe = '
      f"{sweep.loc[best_is, 'oos_sharpe']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sweep.index, sweep['is_sharpe'], marker='o', label='in-sample Sharpe')
ax.plot(sweep.index, sweep['oos_sharpe'], marker='s', label='out-of-sample Sharpe')
ax.axvline(best_is, linestyle='--', alpha=0.5,
           label=f'best IS lookback = {best_is}')
ax.set_title('Parameter sweep: how the in-sample winner really looks out of sample')
ax.set_xlabel('Momentum lookback (periods)')
ax.set_ylabel('Annualized Sharpe')
ax.legend()
plt.show()
print('The peaks of the IS curve are mostly the shape of noise; the OOS curve is closer to the true expectation.')

### A deliberate data-leakage demo (invalid by construction)

The experiment below **cheats on purpose**: it uses the sign of the **current-period** return as the position — which requires knowing the future. It inevitably wins every period and looks absurdly good.

> **This result must never be treated as a strategy.** Its only purpose is to teach you what leakage looks like.

In [ ]:
leaked = leaked_strategy_returns(returns)
leaked_equity = (1 + leaked).cumprod()
honest_equity = result.net_equity

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(leaked_equity.index, leaked_equity.values,
        label='[INVALID] leaked model (peeks at the future)')
ax.plot(honest_equity.index, honest_equity.values,
        label='honest backtest (net)')
ax.set_title('Leakage demo: an absurd curve is a warning sign, not a strategy')
ax.set_xlabel('Date')
ax.set_ylabel('Equity (start = 1)')
ax.legend()
plt.show()
print(f'Leaked total return = {(leaked_equity.iloc[-1] - 1):.2%}  <-- impossible, invalid')
print(f'Honest net total return = {(honest_equity.iloc[-1] - 1):.2%}')

In [ ]:
# Automated leakage check: using the current-period return as a feature gets caught
try:
    assert_no_lookahead(returns, returns, name='current-period return as feature')
    print('No leakage detected')
except ValueError as exc:
    print('Leakage detected:', exc)

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. In your own words, explain the difference between an expanding window and a rolling window.
2. Why must positions driven by a signal be lagged?
3. What is survivorship bias, and why does it make backtests overly optimistic?

### Applied exercises

In [ ]:
# Applied exercise 1: raise trading costs from 5 bps to 20 bps and compare net total returns.
high_cost = None  # TODO: run_backtest(raw_signal, returns, signal_lag=1, cost_per_unit_turnover=0.002)
if high_cost is not None:
    print('20 bps net total return:', high_cost.summary()['total_net_return'])

In [ ]:
# Applied exercise 2: use cost_summary to compare gross vs net total returns and the cost drag.
cs = None  # TODO: cost_summary(result.gross_returns, result.net_returns)
if cs is not None:
    print(cs)

### Reflection question

1. Look back over Weeks 1–7. Where could a great-looking backtest have quietly introduced leakage, overfitting, or multiple-testing problems? List at least three places, and check them against [`docs/common_backtesting_mistakes.md`](../../docs/common_backtesting_mistakes.md).

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. Positions must be lagged at least one period behind the signal because?**
- A. It increases returns
- B. A signal computed only at the close of period t can be traded no sooner than the next period
- C. It reduces trading costs
- D. It makes the curve smoother

**Q2. The difference between expanding and rolling windows is?**
- A. An expanding window has fixed length
- B. A rolling window has fixed length and gradually forgets older data
- C. They are exactly the same
- D. Rolling windows cannot be used on time series

**Q3. The purpose of a purge gap (space between train and test) is?**
- A. Faster computation
- B. Removing leakage from overlapping information at the boundary (e.g. multi-period return labels)
- C. Increasing the sample size
- D. Lowering trading costs

**Q4. After a parameter sweep, the in-sample best parameter usually performs out of sample?**
- A. Just as well
- B. Better
- C. Clearly worse — part of the IS performance was luck
- D. It cannot be computed

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '694d108637f0d0ac', 2: 'c0d8eb5d1a53208a', 3: '99f302aa17fd70bc', 4: '270b78bed9c73640'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w8-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Using random splits instead of time-based splits.**
- **Not lagging positions behind the signal — i.e. peeking at the future (look-ahead bias).**
- **Reporting only gross performance, ignoring trading costs and turnover.**
- **Not comparing against naive baselines (zero return, historical mean, buy-and-hold).**
- **Mistaking the deliberately leaked, absurd result for a real strategy.**
- **Backfilling the leading NaNs of rolling features with future information.**

## After this week, you should be able to

- [ ] Implement time-based splits and walk-forward evaluation.
- [ ] Compare a forecast model honestly against naive baselines.
- [ ] Apply trading costs and compare gross vs net.
- [ ] Recognize leakage and explain why its results are invalid.
- [ ] Complete a small leak-free, cost-aware, reproducible backtesting workflow.

## Closing thoughts

After eight weeks you can go from 'refreshing the mathematical foundations' to 'a small but correct, leakage-proof quantitative research workflow'. Remember the core belief of this roadmap:

**Evaluate correctly before refining models; be reproducible before chasing performance.**

This project is education and research-methodology training — **not investment advice** — and it **claims no** profitable strategy.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.